In [ ]:
import numpy as np
import pandas as pd
from collections import Counter
import random
import math

# ------------------------------------------------------------
# 1. Weather Dataset (14 instances from lecture slides)
# ------------------------------------------------------------
data = [
    ['Sunny', 'Hot', 'High', 'Weak', 'No'],
    ['Sunny', 'Hot', 'High', 'Strong', 'No'],
    ['Cloudy', 'Hot', 'High', 'Weak', 'Yes'],
    ['Rain', 'Mild', 'High', 'Weak', 'Yes'],
    ['Rain', 'Cool', 'Normal', 'Weak', 'Yes'],
    ['Rain', 'Cool', 'Normal', 'Strong', 'No'],
    ['Cloudy', 'Cool', 'Normal', 'Strong', 'Yes'],
    ['Sunny', 'Mild', 'High', 'Weak', 'No'],
    ['Sunny', 'Cool', 'Normal', 'Weak', 'Yes'],
    ['Rain', 'Mild', 'Normal', 'Weak', 'Yes'],
    ['Sunny', 'Mild', 'Normal', 'Strong', 'Yes'],
    ['Cloudy', 'Mild', 'High', 'Strong', 'Yes'],
    ['Cloudy', 'Hot', 'Normal', 'Weak', 'Yes'],
    ['Rain', 'Mild', 'High', 'Strong', 'No']
]

columns = ['Outlook', 'Temperature', 'Humidity', 'Wind', 'Play']
df = pd.DataFrame(data, columns=columns)
print("Dataset shape:", df.shape)
print(df.head())

# ------------------------------------------------------------
# 2. Helper functions for entropy and information gain
# ------------------------------------------------------------
def entropy(labels):
    """Calculate entropy of a list of labels."""
    counts = Counter(labels)
    probs = [count / len(labels) for count in counts.values()]
    return -sum(p * math.log2(p) for p in probs if p > 0)

def information_gain(parent_labels, splits):
    """
    splits: list of child label lists
    Returns information gain = entropy(parent) - weighted_avg_entropy(children)
    """
    parent_entropy = entropy(parent_labels)
    total = len(parent_labels)
    weighted_child_entropy = sum((len(child)/total) * entropy(child) for child in splits)
    return parent_entropy - weighted_child_entropy

# ------------------------------------------------------------
# 3. Decision Tree from scratch
# ------------------------------------------------------------
class Node:
    def __init__(self, attribute=None, value=None, children=None, leaf_label=None):
        self.attribute = attribute      # feature name for split
        self.value = value              # split value (for categorical)
        self.children = children        # dict: {value_of_feature: Node}
        self.leaf_label = leaf_label    # if leaf, final prediction

def build_tree(data, features, max_depth=3, min_samples_split=2, depth=0):
    """
    Recursively build decision tree using information gain.
    data: pandas DataFrame with last column as target 'Play'
    features: list of feature names (excluding target)
    """
    labels = data['Play'].tolist()

    # Stopping conditions
    if len(set(labels)) == 1:           # pure node
        return Node(leaf_label=labels[0])
    if depth >= max_depth:
        return Node(leaf_label=Counter(labels).most_common(1)[0][0])
    if len(data) < min_samples_split:
        return Node(leaf_label=Counter(labels).most_common(1)[0][0])
    if not features:
        return Node(leaf_label=Counter(labels).most_common(1)[0][0])

    # Find best feature and split
    best_gain = -1
    best_feature = None
    best_splits = None
    best_split_values = None

    for feature in features:
        # For categorical features, split on each unique value
        values = data[feature].unique()
        splits = []
        split_values = []
        for val in values:
            subset = data[data[feature] == val]
            splits.append(subset['Play'].tolist())
            split_values.append(val)
        gain = information_gain(labels, splits)
        if gain > best_gain:
            best_gain = gain
            best_feature = feature
            best_splits = splits
            best_split_values = values

    if best_gain == 0:   # no improvement
        return Node(leaf_label=Counter(labels).most_common(1)[0][0])

    # Build children recursively
    children = {}
    remaining_features = [f for f in features if f != best_feature]
    for val, split_labels in zip(best_split_values, best_splits):
        subset = data[data[best_feature] == val]
        child_node = build_tree(subset, remaining_features, max_depth, min_samples_split, depth+1)
        children[val] = child_node

    return Node(attribute=best_feature, children=children)

def predict_tree(node, instance):
    """Predict single instance: dictionary of {feature: value}"""
    if node.leaf_label is not None:
        return node.leaf_label
    feature_value = instance[node.attribute]
    if feature_value not in node.children:
        # if unseen value, return majority (fallback) – here just pick first child
        return next(iter(node.children.values())).leaf_label
    return predict_tree(node.children[feature_value], instance)

# ------------------------------------------------------------
# 4. Random Forest from scratch
# ------------------------------------------------------------
def bootstrap_sample(data):
    """Create a bootstrap dataset by sampling with replacement."""
    n = len(data)
    indices = np.random.choice(n, n, replace=True)
    return data.iloc[indices].reset_index(drop=True)

def random_feature_subset(features, k=None):
    """Randomly select k features (default sqrt)."""
    if k is None:
        k = int(math.sqrt(len(features))) + 1
    return random.sample(features, k)

class RandomForest:
    def __init__(self, n_trees=10, max_depth=3, min_samples_split=2, feature_subset_size=None):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.feature_subset_size = feature_subset_size
        self.trees = []

    def fit(self, df):
        features = df.columns[:-1].tolist()   # all except 'Play'
        for _ in range(self.n_trees):
            # Bootstrap sample
            bootstrap_df = bootstrap_sample(df)
            # Random feature subset
            feat_subset = random_feature_subset(features, self.feature_subset_size)
            # Build tree on bootstrap with subset of features
            tree = build_tree(bootstrap_df, feat_subset, self.max_depth, self.min_samples_split)
            self.trees.append(tree)

    def predict(self, instances_df):
        """Predict for DataFrame of instances."""
        predictions = []
        for _, row in instances_df.iterrows():
            instance = row.to_dict()
            tree_preds = [predict_tree(tree, instance) for tree in self.trees]
            majority = Counter(tree_preds).most_common(1)[0][0]
            predictions.append(majority)
        return predictions

# ------------------------------------------------------------
# 5. Evaluation function
# ------------------------------------------------------------
def evaluate(model, X_test, y_test, model_name="Model"):
    y_pred = model.predict(X_test) if hasattr(model, 'predict') else [predict_tree(model, row.to_dict()) for _, row in X_test.iterrows()]
    acc = sum(1 for i in range(len(y_test)) if y_pred[i] == y_test.iloc[i]) / len(y_test)
    print(f"{model_name} Accuracy: {acc:.4f}")
    return acc

# ------------------------------------------------------------
# 6. Train/Test split (leave-one-out for small dataset, but here 70/30)
# ------------------------------------------------------------
from sklearn.model_selection import train_test_split
X = df.drop('Play', axis=1)
y = df['Play']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# ------------------------------------------------------------
# 7. Train Decision Tree from scratch
# ------------------------------------------------------------
features = X.columns.tolist()
train_df = X_train.copy()
train_df['Play'] = y_train

decision_tree = build_tree(train_df, features, max_depth=5, min_samples_split=2)
print("\n" + "="*50)
print("Decision Tree (from scratch)")
evaluate(decision_tree, X_test, y_test, "Decision Tree")

# ------------------------------------------------------------
# 8. Train Random Forest from scratch
# ------------------------------------------------------------
rf = RandomForest(n_trees=15, max_depth=4, min_samples_split=2, feature_subset_size=2)
rf.fit(train_df)
print("\nRandom Forest (from scratch)")
evaluate(rf, X_test, y_test, "Random Forest")

# ------------------------------------------------------------
# 9. Compare with manual prediction on a single instance
# ------------------------------------------------------------
print("\n" + "="*50)
print("Example prediction for new instance:")
new_instance = pd.DataFrame([['Sunny', 'Cool', 'High', 'Strong']], columns=columns[:-1])
print("New instance: Sunny, Cool, High, Strong")
dt_pred = predict_tree(decision_tree, new_instance.iloc[0].to_dict())
rf_pred = rf.predict(new_instance)[0]
print(f"Decision Tree predicts: {dt_pred}")
print(f"Random Forest predicts: {rf_pred}")

# ------------------------------------------------------------
# 10. Feature importance from Random Forest (simple count)
# ------------------------------------------------------------
def rf_feature_importance(rf_model, features):
    importance = {f: 0 for f in features}
    for tree in rf_model.trees:
        # traverse tree and count attribute usage
        stack = [tree]
        while stack:
            node = stack.pop()
            if node.attribute is not None:
                importance[node.attribute] += 1
                for child in node.children.values():
                    stack.append(child)
    total = sum(importance.values())
    return {k: v/total for k, v in importance.items()}

importance = rf_feature_importance(rf, features)
print("\nRandom Forest Feature Importance (based on split frequency):")
for feat, imp in sorted(importance.items(), key=lambda x: x[1], reverse=True):
    print(f"  {feat}: {imp:.3f}")

Dataset shape: (14, 5)
  Outlook Temperature Humidity    Wind Play
0   Sunny         Hot     High    Weak   No
1   Sunny         Hot     High  Strong   No
2  Cloudy         Hot     High    Weak  Yes
3    Rain        Mild     High    Weak  Yes
4    Rain        Cool   Normal    Weak  Yes

Decision Tree (from scratch)
Decision Tree Accuracy: 0.8000

Random Forest (from scratch)
Random Forest Accuracy: 0.8000

Example prediction for new instance:
New instance: Sunny, Cool, High, Strong
Decision Tree predicts: No
Random Forest predicts: Yes

Random Forest Feature Importance (based on split frequency):
  Outlook: 0.448
  Temperature: 0.241
  Humidity: 0.241
  Wind: 0.069
